# HybridAgent — Exploración interactiva

Este notebook carga el paclet HVA y recorre las funciones principales de `HybridAgent`:
`AgentId`, accessors, actualizaciones inmutables y hash estructural.

## 1 · Cargar el paclet

In [1]:
PacletDirectoryLoad["/workspaces/hva-framework/paclet"];
Needs["HVA`"]

## 2 · Construir un agente de prueba

In [5]:
(* Agente termostat mínimo *)
Spec = HybridAgent["thermostat",
  Modes            -> {"off", "on"},
  ContinuousVars   -> {temp},
  VectorFields     -> <|
    "off" -> {Derivative[1][temp] == 0},
    "on"  -> {Derivative[1][temp] == 5 - temp}
  |>,
  Transitions      -> {
    <|"from" -> "off", "to" -> "on",  "condition" -> temp < 18|>,
    <|"from" -> "on",  "to" -> "off", "condition" -> temp > 22|>
  },
  ModeInvariants   -> {0 <= temp <= 30},
  InitialMode      -> "off",
  InitialValuation -> <|temp -> 15|>
]

HybridAgent["thermostat" @ off]

In [ ]:
(* Verificar que se construyó correctamente *)
HybridAgentQ[Spec]

True

## 3 · `AgentId` — identificador del agente

```
AgentId::usage = "AgentId[a] devuelve el identificador del agente."
```

In [6]:
AgentId[Spec]

thermostat

In [7]:
#(* Verificar tipo de retorno *)
StringQ[AgentId[Spec]]

True

In [20]:
#(* Error handling: non-HybridAgent *)
AgentId["not an agent"]

Expected HybridAgent.

## 4 · Accessors completos

In [7]:
AgentModes[Spec]

AgentStates[Unknown option(s) provided to HybridAgent.]

In [8]:
AgentContinuousVars[Spec]

AgentVars[Unknown option(s) provided to HybridAgent.]

In [9]:
AgentVectorFields[Spec]

AgentDynamics[Unknown option(s) provided to HybridAgent.]

In [10]:
AgentTransitions[Spec]

AgentGuards[Unknown option(s) provided to HybridAgent.]

In [11]:
AgentCurrentMode[Spec]

AgentCurrentState[Unknown option(s) provided to HybridAgent.]

In [12]:
AgentValuation[Spec]

Expected HybridAgent.

In [13]:
AgentValuation[Spec][temp]

Missing[NotAvailable, temp]

## 5 · Actualizaciones inmutables

Cada `With*` devuelve un **nuevo** agente sin modificar el original.

In [14]:
agent2 = WithCurrentMode[Spec, "on"];
{AgentCurrentMode[Spec], AgentCurrentMode[agent2]}

>    AgentCurrentState[WithCurrentState[Unknown option(s) provided to\
>       HybridAgent., on]]}


{AgentCurrentState[Unknown option(s) provided to HybridAgent.],

In [15]:
Spec3 = WithValuation[agent2, <|temp -> 20|>];
AgentValuation[Spec3][temp]

Missing[NotAvailable, temp]

In [16]:
Spec4 = AppendTrace[Spec3, "transitioned to on"];
AgentTrace[Spec4]

Expected HybridAgent.

## 6 · Hash estructural

`AgentStructuralHash` es determinístico e ignora campos runtime (`currentState`, `valuation`, `mailbox`, `trace`).

In [17]:
#(* Mismo hash en dos instancias con distintos runtime fields *)
h1 = AgentStructuralHash[Spec3];
h2 = AgentStructuralHash[Spec4];
{h1, h2, h1 === h2}

>     <|Function -> AgentStructuralHash, Expected -> HybridAgent, 
>      Got -> Failure|>], Failure[HVAArgumentError, 
>     <|Function -> AgentStructuralHash, Expected -> HybridAgent, 
>      Got -> Failure|>], True}


{Failure[HVAArgumentError,

In [18]:
#(* Hash cambia si se modifica la estructura *)
SpecMod = HybridAgent["thermostat",
  Modes            -> {"off", "on", "standby"},
  ContinuousVars   -> {temp},
  VectorFields     -> <|
    "off"     -> {Derivative[1][temp] == 0},
    "on"      -> {Derivative[1][temp] == 5 - temp},
    "standby" -> {Derivative[1][temp] == -0.1 * temp}
  |>,
  Transitions      -> {},
  ModeInvariants   -> {},
  InitialMode      -> "off",
  InitialValuation -> <|temp -> 15|>
];
AgentStructuralHash[Spec3] =!= AgentStructuralHash[SpecMod]

False